In [3]:
!pip install torch transformers sentencepiece ctranslate2 huggingface_hub
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import sentencepiece as spm
from ctranslate2 import Translator
model = AutoModelForCausalLM.from_pretrained("CraneAILabs/swahili-gemma-1b")
tokenizer = AutoTokenizer.from_pretrained("CraneAILabs/swahili-gemma-1b")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 47.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [1]:
eng_to_swa_prompt = input("Translate to swa: ")
swa_to_eng_prompt = input("Translate to eng: ")

Translate to swa:  Although it was raining, we decided to play football because it was the last day of school.
Translate to eng:  ...


In [5]:
# English to Swahili translation
path_to_model = "/kaggle/input/models/jaydenmunene/en-swa1/transformers/default/1/en-swa"
source = 'en'
target = 'sw'

translator = Translator(path_to_model, compute_type='int8')
source_tokenizer = spm.SentencePieceProcessor(f'{path_to_model}/{source}.spm.model')
target_tokenizer = spm.SentencePieceProcessor(f'{path_to_model}/{target}.spm.model')

text = eng_to_swa_prompt

input_tokens = [source_tokenizer.EncodeAsPieces(text)]
translator_output = translator.translate_batch(
  input_tokens,
  batch_type='tokens',
  beam_size=2,
  max_input_length=0,
  max_decoding_length=128
)

output_tokens = [item.hypotheses[0] for item in translator_output]
translation = target_tokenizer.DecodePieces(output_tokens)

In [8]:
# Swahili to English translation

path_to_model = "/kaggle/input/models/jaydenmunene/swa-en/transformers/default/1/swa-en"
source = 'sw'
target = 'en'

translator = Translator(path_to_model, compute_type='int8')
source_tokenizer = spm.SentencePieceProcessor(f'{path_to_model}/{source}.spm.model')
target_tokenizer = spm.SentencePieceProcessor(f'{path_to_model}/{target}.spm.model')

text = swa_to_eng_prompt

input_tokens = [source_tokenizer.EncodeAsPieces(text)]
translator_output = translator.translate_batch(
  input_tokens,
  batch_type='tokens',
  beam_size=4,
  max_input_length=0,
  max_decoding_length=128
)

output_tokens = [item.hypotheses[0] for item in translator_output]
translation_to_eng = target_tokenizer.DecodePieces(output_tokens)

In [13]:
# Results
print("Translation in Swahili: ", translation)
print("Translation in English: ", translation_to_eng)

Translation in Swahili:  ['Ingawa mvua ilikuwa ikinyesha, tuliamua kucheza soka kwa sababu ilikuwa siku ya mwisho shuleni.']
Translation in English:  ['...']
